# LTB Evidence Brief Generator

**Works in:** Google Colab · Local Jupyter

### Quick start
1. Run **Cell 1** to install dependencies (one-time).
2. Run **Cell 2** to create the required directories.
3. Run **Cell 3** to write `app.py`.
4. Run **Cell 4** to write `templates/index.html`.
5. Run **Cell 5** to start the web server and get your URL.

> **Colab note:** the URL shown is a temporary proxy link — bookmark it for the session.
> Re-run Cell 5 at any time to retrieve the URL again.


In [ ]:
# Cell 1 — Install dependencies
!pip install flask werkzeug pillow reportlab pdfplumber python-docx openpyxl -q
print('Dependencies installed.')


In [ ]:
# Cell 2 — Create required directories
import os
os.makedirs('templates', exist_ok=True)
os.makedirs('uploads', exist_ok=True)
print('Directories ready.')


In [ ]:
%%writefile app.py
# -*- coding: utf-8 -*-
import os
import json
import uuid
import io
from xml.sax.saxutils import escape as xml_escape
from flask import Flask, request, jsonify, send_file, render_template
from werkzeug.utils import secure_filename
from PIL import Image
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, PageBreak,
    Image as RLImage, Table, TableStyle, HRFlowable
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT
from reportlab.pdfgen import canvas
from reportlab.platypus.flowables import Flowable

# Optional text-extraction dependencies (app works without them)
try:
    import pdfplumber
    HAS_PDFPLUMBER = True
except ImportError:
    HAS_PDFPLUMBER = False

try:
    from docx import Document as DocxDocument
    HAS_DOCX = True
except ImportError:
    HAS_DOCX = False

try:
    import openpyxl
    HAS_OPENPYXL = True
except ImportError:
    HAS_OPENPYXL = False

app = Flask(__name__)
app.config['UPLOAD_FOLDER'] = os.path.join(os.path.dirname(__file__), 'uploads')
app.config['MAX_CONTENT_LENGTH'] = 50 * 1024 * 1024  # 50MB

IMAGE_EXTS = {'png', 'jpg', 'jpeg', 'gif', 'bmp', 'tiff', 'webp'}
PLAIN_TEXT_EXTS = {
    'txt', 'md', 'csv', 'log', 'py', 'js', 'ts', 'html', 'htm',
    'css', 'json', 'xml', 'yaml', 'yml', 'ini', 'cfg', 'conf',
    'sql', 'sh', 'bat', 'tex', 'rst', 'rtf',
}

def is_image_file(path):
    ext = path.rsplit('.', 1)[-1].lower() if '.' in path else ''
    return ext in IMAGE_EXTS


def extract_text_from_file(path, max_chars=4000):
    """Return (text, truncated, success). Reads up to max_chars of extractable text."""
    ext = path.rsplit('.', 1)[-1].lower() if '.' in path else ''
    try:
        if ext in PLAIN_TEXT_EXTS:
            with open(path, 'r', encoding='utf-8', errors='replace') as f:
                raw = f.read(max_chars + 100)
            truncated = len(raw) > max_chars
            return raw[:max_chars], truncated, True

        elif ext == 'pdf':
            if not HAS_PDFPLUMBER:
                return None, False, False
            parts = []
            with pdfplumber.open(path) as pdf:
                for page in pdf.pages:
                    parts.append(page.extract_text() or '')
                    if sum(len(p) for p in parts) >= max_chars + 100:
                        break
            raw = '\n'.join(parts)
            truncated = len(raw) > max_chars
            return raw[:max_chars], truncated, True

        elif ext == 'docx':
            if not HAS_DOCX:
                return None, False, False
            doc = DocxDocument(path)
            raw = '\n'.join(p.text for p in doc.paragraphs if p.text)
            truncated = len(raw) > max_chars
            return raw[:max_chars], truncated, True

        elif ext in ('xlsx', 'xls'):
            if not HAS_OPENPYXL:
                return None, False, False
            wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
            parts = []
            for sheet_name in wb.sheetnames:
                ws = wb[sheet_name]
                parts.append("── Sheet: %s ──" % sheet_name)
                for row in ws.iter_rows(values_only=True):
                    cells = [str(c) if c is not None else '' for c in row]
                    parts.append('\t'.join(cells))
                    if sum(len(p) for p in parts) >= max_chars + 100:
                        break
                if sum(len(p) for p in parts) >= max_chars + 100:
                    break
            wb.close()
            raw = '\n'.join(parts)
            truncated = len(raw) > max_chars
            return raw[:max_chars], truncated, True

        else:
            return None, False, False

    except Exception:
        return None, False, False


# ─── PAGE NUMBERING ────────────────────────────────────────────────────────────

class NumberedCanvas(canvas.Canvas):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._saved_page_states = []

    def showPage(self):
        self._saved_page_states.append(dict(self.__dict__))
        self._startPage()

    def save(self):
        for state in self._saved_page_states:
            self.__dict__.update(state)
            self.draw_page_footer()
            super().showPage()
        super().save()

    def draw_page_footer(self):
        self.setFont("Helvetica", 8)
        self.setFillColor(colors.HexColor("#666666"))
        width, height = letter
        self.drawRightString(width - 0.5 * inch, 0.4 * inch, "Page %d" % self._pageNumber)
        self.drawString(0.5 * inch, 0.4 * inch, "Evidence Brief")
        self.setStrokeColor(colors.HexColor("#cccccc"))
        self.setLineWidth(0.5)
        self.line(0.5 * inch, 0.55 * inch, width - 0.5 * inch, 0.55 * inch)


# ─── TAB DIVIDER FLOWABLE ──────────────────────────────────────────────────────

class TabDivider(Flowable):
    """Full-page coloured tab divider.
    If page_registry is supplied, records self.canv._pageNumber keyed by registry_key
    on first draw (used for TOC page-number two-pass rendering).
    """
    def __init__(self, tab_number, tab_title, width, height,
                 page_registry=None, registry_key=None):
        super().__init__()
        self.tab_number = tab_number
        self.tab_title = tab_title
        self.width = width
        self.height = height
        self.page_registry = page_registry
        self.registry_key = registry_key if registry_key is not None else tab_number

    def draw(self):
        # Record page number for TOC (pass 1 only)
        if self.page_registry is not None:
            self.page_registry[self.registry_key] = self.canv._pageNumber

        tab_color = colors.HexColor("#1a3a5c")
        self.canv.setFillColor(tab_color)
        self.canv.rect(
            self.width - 1.5 * inch, -0.5 * inch,
            1.5 * inch, self.height + 1 * inch,
            fill=1, stroke=0
        )
        self.canv.setFillColor(colors.white)
        self.canv.setFont("Helvetica-Bold", 14)
        self.canv.saveState()
        self.canv.translate(self.width - 0.75 * inch, self.height / 2)
        self.canv.rotate(270)
        label = "TAB %d" % self.tab_number
        if self.tab_title:
            label += "  -  " + self.tab_title
        self.canv.drawCentredString(0, 0, label)
        self.canv.restoreState()

        self.canv.setFillColor(colors.HexColor("#1a3a5c"))
        self.canv.setFont("Helvetica-Bold", 72)
        self.canv.drawCentredString(
            (self.width - 1.5 * inch) / 2,
            self.height / 2 - 0.25 * inch,
            str(self.tab_number)
        )
        if self.tab_title:
            self.canv.setFont("Helvetica", 18)
            self.canv.drawCentredString(
                (self.width - 1.5 * inch) / 2,
                self.height / 2 - 0.9 * inch,
                self.tab_title
            )

    def wrap(self, availWidth, availHeight):
        return self.width, self.height


# ─── STORY BUILDER ─────────────────────────────────────────────────────────────

def _build_story(case_info, tabs_data, usable_w, usable_h,
                 page_registry=None, tab_pages=None, text_cache=None):
    """
    Build and return the full ReportLab story.

    page_registry  – dict populated with {tab_idx: page_number} during draw (pass 1).
    tab_pages      – dict {tab_idx: page_number} used to fill the TOC (pass 2).
    text_cache     – shared dict {path: (text, truncated, ok)} to avoid double extraction.
    """
    if text_cache is None:
        text_cache = {}

    styles = getSampleStyleSheet()
    story = []
    navy = colors.HexColor("#1a3a5c")
    gold = colors.HexColor("#c8a951")

    # ── TITLE PAGE ────────────────────────────────────────────────────────────
    story.append(Spacer(1, 1.5 * inch))

    title_style = ParagraphStyle(
        'BriefTitle', parent=styles['Normal'],
        fontName='Helvetica-Bold', fontSize=36,
        textColor=navy, alignment=TA_CENTER,
        spaceAfter=0,
    )
    story.append(Paragraph("Evidence Brief", title_style))
    story.append(Spacer(1, 0.28 * inch))
    story.append(HRFlowable(width="100%", thickness=3, color=gold))
    story.append(Spacer(1, 0.4 * inch))

    label_style = ParagraphStyle(
        'Label', parent=styles['Normal'],
        fontName='Helvetica-Bold', fontSize=10,
        textColor=navy, spaceBefore=4,
    )
    value_style = ParagraphStyle(
        'Value', parent=styles['Normal'],
        fontName='Helvetica', fontSize=11,
        spaceBefore=2,
    )

    def row(label, value):
        return [Paragraph(label, label_style), Paragraph(value or '—', value_style)]

    case_table_data = [
        row("FILE NUMBER",       case_info.get('file_number', '')),
        row("HEARING DATE",      case_info.get('hearing_date', '')),
        row("APPLICANT",         case_info.get('applicant_name', '')),
        row("APPLICANT ADDRESS", case_info.get('applicant_address', '')),
        row("RESPONDENT",        case_info.get('respondent_name', '')),
        row("RESPONDENT ADDRESS",case_info.get('respondent_address', '')),
    ]
    col_w = [1.8 * inch, usable_w - 1.8 * inch]
    case_table = Table(case_table_data, colWidths=col_w)
    case_table.setStyle(TableStyle([
        ('VALIGN',        (0, 0), (-1, -1), 'TOP'),
        ('ROWBACKGROUNDS',(0, 0), (-1, -1), [colors.HexColor("#f5f7fa"), colors.white]),
        ('TOPPADDING',    (0, 0), (-1, -1), 8),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 8),
        ('LEFTPADDING',   (0, 0), (-1, -1), 10),
        ('RIGHTPADDING',  (0, 0), (-1, -1), 10),
        ('LINEBELOW',     (0, -1), (-1, -1), 0.5, colors.HexColor("#cccccc")),
    ]))
    story.append(case_table)
    story.append(Spacer(1, 0.5 * inch))
    story.append(HRFlowable(width="100%", thickness=1, color=colors.HexColor("#cccccc")))
    story.append(Spacer(1, 0.2 * inch))

    tab_summary_style = ParagraphStyle(
        'TabSummary', parent=styles['Normal'],
        fontName='Helvetica', fontSize=10,
        textColor=colors.HexColor("#555555"),
        alignment=TA_CENTER,
    )
    plural = 's' if len(tabs_data) != 1 else ''
    story.append(Paragraph(
        "This brief contains <b>%d tab%s</b>." % (len(tabs_data), plural),
        tab_summary_style
    ))
    story.append(PageBreak())

    # ── TABLE OF CONTENTS ─────────────────────────────────────────────────────
    toc_title_style = ParagraphStyle(
        'TOCTitle', parent=styles['Title'],
        fontName='Helvetica-Bold', fontSize=18,
        textColor=navy, spaceAfter=16, spaceBefore=0,
    )
    story.append(Paragraph("Table of Contents", toc_title_style))
    story.append(HRFlowable(width="100%", thickness=2, color=gold))
    story.append(Spacer(1, 0.25 * inch))

    toc_entry_style = ParagraphStyle(
        'TOCEntry', parent=styles['Normal'],
        fontName='Helvetica', fontSize=12, spaceAfter=6,
    )
    toc_header_style = ParagraphStyle(
        'TOCHeader', parent=styles['Normal'],
        fontName='Helvetica-Bold', fontSize=12,
        spaceAfter=6, textColor=navy,
    )
    toc_page_style = ParagraphStyle(
        'TOCPage', parent=styles['Normal'],
        fontName='Helvetica', fontSize=12,
        spaceAfter=6, alignment=TA_RIGHT,
    )

    toc_data = [[
        Paragraph("<b>Tab</b>",         toc_header_style),
        Paragraph("<b>Description</b>", toc_header_style),
        Paragraph("<b>Page</b>",        toc_header_style),
    ]]
    for i, tab in enumerate(tabs_data, 1):
        pg = str(tab_pages[i]) if (tab_pages and i in tab_pages) else '—'
        toc_data.append([
            Paragraph(str(i), toc_entry_style),
            Paragraph(tab.get('title') or ("Tab %d" % i), toc_entry_style),
            Paragraph(pg, toc_page_style),
        ])

    toc_table = Table(toc_data, colWidths=[0.6 * inch, usable_w - 1.6 * inch, 1 * inch])
    toc_table.setStyle(TableStyle([
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.HexColor("#f5f7fa"), colors.white]),
        ('TOPPADDING',     (0, 0), (-1, -1), 8),
        ('BOTTOMPADDING',  (0, 0), (-1, -1), 8),
        ('LEFTPADDING',    (0, 0), (-1, -1), 10),
        ('LINEBELOW',      (0, 0), (-1,  0), 1,   navy),
        ('LINEBELOW',      (0, -1), (-1, -1), 0.5, colors.HexColor("#cccccc")),
        ('VALIGN',         (0, 0), (-1, -1), 'MIDDLE'),
        ('ALIGN',          (2, 0), (2,  -1), 'RIGHT'),
    ]))
    story.append(toc_table)
    story.append(PageBreak())

    # ── TAB SECTIONS ──────────────────────────────────────────────────────────
    photo_caption_style = ParagraphStyle(
        'PhotoCaption', parent=styles['Normal'],
        fontName='Helvetica', fontSize=9,
        textColor=colors.HexColor("#666666"),
        alignment=TA_CENTER, spaceAfter=16,
    )
    no_file_style = ParagraphStyle(
        'NoFile', parent=styles['Normal'],
        fontName='Helvetica', fontSize=11,
        textColor=colors.grey, alignment=TA_CENTER,
    )
    doc_header_style = ParagraphStyle(
        'DocHeader', parent=styles['Normal'],
        fontName='Helvetica-Bold', fontSize=11,
        textColor=navy, spaceBefore=10, spaceAfter=6,
    )
    doc_body_style = ParagraphStyle(
        'DocBody', parent=styles['Normal'],
        fontName='Helvetica', fontSize=9,
        leading=13, spaceAfter=4,
    )
    doc_note_style = ParagraphStyle(
        'DocNote', parent=styles['Normal'],
        fontName='Helvetica-Oblique', fontSize=8,
        textColor=colors.HexColor("#888888"), spaceAfter=8,
    )

    for tab_idx, tab in enumerate(tabs_data, 1):
        story.append(TabDivider(
            tab_number=tab_idx,
            tab_title=tab.get('title', ''),
            width=usable_w,
            height=usable_h,
            page_registry=page_registry,
            registry_key=tab_idx,
        ))
        story.append(PageBreak())

        files = tab.get('images', [])
        if not files:
            story.append(Spacer(1, 2 * inch))
            story.append(Paragraph("No files uploaded for this tab.", no_file_style))
            story.append(PageBreak())
            continue

        img_files = [(i, p) for i, p in enumerate(files) if is_image_file(p)]
        doc_files = [(i, p) for i, p in enumerate(files) if not is_image_file(p)]

        # Embed images 2 per page
        img_counter = 0
        for i in range(0, len(img_files), 2):
            page_imgs = img_files[i:i+2]
            for orig_idx, img_path in page_imgs:
                img_counter += 1
                try:
                    with Image.open(img_path) as pil_img:
                        orig_w, orig_h = pil_img.size
                    max_w = usable_w
                    max_h = (usable_h - 0.5 * inch) / 2
                    scale = min(max_w / orig_w, max_h / orig_h, 1.0)
                    rl_img = RLImage(img_path,
                                     width=orig_w * scale,
                                     height=orig_h * scale)
                    rl_img.hAlign = 'CENTER'
                    story.append(rl_img)
                    title_part = (" - " + tab.get('title')) if tab.get('title') else ""
                    story.append(Paragraph(
                        "Tab %d%s | Photo %d" % (tab_idx, title_part, img_counter),
                        photo_caption_style
                    ))
                    story.append(Spacer(1, 0.15 * inch))
                except Exception:
                    story.append(Paragraph(
                        "[Image could not be loaded: %s]" % os.path.basename(img_path),
                        photo_caption_style
                    ))
            story.append(PageBreak())

        # Non-image documents: extract and display text content
        for orig_idx, doc_path in doc_files:
            fname = os.path.basename(doc_path)
            ext = fname.rsplit('.', 1)[-1].upper() if '.' in fname else 'FILE'

            # File header
            story.append(Paragraph("[%s]  %s" % (ext, fname), doc_header_style))
            story.append(HRFlowable(
                width="100%", thickness=0.5,
                color=colors.HexColor("#cccccc")
            ))
            story.append(Spacer(1, 0.1 * inch))

            # Cached text extraction
            if doc_path not in text_cache:
                text_cache[doc_path] = extract_text_from_file(doc_path)
            text, truncated, ok = text_cache[doc_path]

            if ok and text and text.strip():
                # Normalise line endings, escape XML, convert newlines to <br/>
                clean = text.replace('\r\n', '\n').replace('\r', '\n')
                safe  = xml_escape(clean)
                html  = safe.replace('\n\n', '<br/><br/>').replace('\n', '<br/>')
                story.append(Paragraph(html, doc_body_style))
                if truncated:
                    story.append(Paragraph(
                        "— content truncated at 4,000 characters —",
                        doc_note_style
                    ))
            elif not ok:
                story.append(Paragraph(
                    "Binary or unsupported format — install pdfplumber / python-docx / "
                    "openpyxl for PDF, DOCX, and Excel extraction.",
                    doc_note_style
                ))
            else:
                story.append(Paragraph(
                    "[File appears to be empty or contains no extractable text.]",
                    doc_note_style
                ))

            story.append(Spacer(1, 0.3 * inch))

        if doc_files:
            story.append(PageBreak())

    return story


# ─── PDF GENERATION (two-pass for accurate TOC page numbers) ──────────────────

def generate_evidence_brief(case_info, tabs_data, output_path):
    page_w, page_h = letter
    doc_kwargs = dict(
        pagesize=letter,
        leftMargin=0.75 * inch,
        rightMargin=0.75 * inch,
        topMargin=0.75 * inch,
        bottomMargin=0.75 * inch,
    )
    usable_w = page_w - 1.5 * inch
    usable_h = page_h - 1.5 * inch - 0.5 * inch

    # Shared text cache so documents are only read once across both passes
    text_cache = {}

    # Pass 1 — dry run to BytesIO; populates page_registry via TabDivider.draw()
    page_registry = {}
    doc1 = SimpleDocTemplate(io.BytesIO(), **doc_kwargs)
    story1 = _build_story(case_info, tabs_data, usable_w, usable_h,
                          page_registry=page_registry,
                          tab_pages=None,
                          text_cache=text_cache)
    doc1.build(story1, canvasmaker=NumberedCanvas)

    # Pass 2 — real render with accurate TOC page numbers
    doc2 = SimpleDocTemplate(output_path, **doc_kwargs)
    story2 = _build_story(case_info, tabs_data, usable_w, usable_h,
                          page_registry=None,
                          tab_pages=page_registry,
                          text_cache=text_cache)
    doc2.build(story2, canvasmaker=NumberedCanvas)


# ─── ROUTES ───────────────────────────────────────────────────────────────────

@app.route('/')
def index():
    return render_template('index.html')


@app.route('/upload', methods=['POST'])
def upload_files():
    if 'files' not in request.files:
        return jsonify({'error': 'No files provided'}), 400

    saved = []
    for f in request.files.getlist('files'):
        if not f or not f.filename:
            continue
        original_name = f.filename
        ext = original_name.rsplit('.', 1)[-1].lower() if '.' in original_name else 'bin'
        fname = "%s.%s" % (uuid.uuid4().hex, ext)
        path = os.path.join(app.config['UPLOAD_FOLDER'], fname)
        f.save(path)
        thumb_name = None
        is_image = False
        try:
            with Image.open(path) as img:
                img.thumbnail((300, 300))
                thumb_name = "thumb_%s" % fname
                thumb_path = os.path.join(app.config['UPLOAD_FOLDER'], thumb_name)
                img.save(thumb_path)
                is_image = True
        except Exception:
            pass
        saved.append({'id': fname, 'thumb': thumb_name, 'original': fname,
                      'name': original_name, 'is_image': is_image})

    return jsonify({'files': saved})


@app.route('/thumbnail/<filename>')
def get_thumbnail(filename):
    path = os.path.join(app.config['UPLOAD_FOLDER'], filename)
    if os.path.exists(path):
        return send_file(path)
    return '', 404


@app.route('/generate', methods=['POST'])
def generate():
    data = request.get_json()
    if not data:
        return jsonify({'error': 'No data provided'}), 400

    case_info = data.get('case_info', {})
    tabs_data = data.get('tabs', [])

    for tab in tabs_data:
        resolved = []
        for img_id in tab.get('images', []):
            path = os.path.join(app.config['UPLOAD_FOLDER'], img_id)
            if os.path.exists(path):
                resolved.append(path)
        tab['images'] = resolved

    output_path = os.path.join(
        app.config['UPLOAD_FOLDER'], "brief_%s.pdf" % uuid.uuid4().hex
    )

    try:
        generate_evidence_brief(case_info, tabs_data, output_path)
        download_name = "Evidence_Brief_%s.pdf" % case_info.get('file_number', 'LTB')
        return send_file(
            output_path,
            mimetype='application/pdf',
            as_attachment=True,
            download_name=download_name,
        )
    except Exception as e:
        return jsonify({'error': str(e)}), 500


if __name__ == '__main__':
    os.makedirs(app.config['UPLOAD_FOLDER'], exist_ok=True)
    app.run(debug=True, port=5050)


In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>LTB Evidence Brief Generator</title>
<style>
  @import url('https://fonts.googleapis.com/css2?family=Source+Serif+4:ital,opsz,wght@0,8..60,300;0,8..60,400;0,8..60,600;1,8..60,400&family=DM+Sans:wght@300;400;500;600&display=swap');

  :root {
    --navy: #1a3a5c;
    --navy-light: #2a5080;
    --gold: #c8a951;
    --gold-light: #e8c96a;
    --bg: #f4f6f9;
    --surface: #ffffff;
    --border: #d8dfe8;
    --text: #1e2b3a;
    --text-light: #6b7e94;
    --danger: #c0392b;
    --success: #27ae60;
    --radius: 10px;
    --shadow: 0 2px 12px rgba(26,58,92,0.10);
  }

  * { box-sizing: border-box; margin: 0; padding: 0; }

  body {
    font-family: 'DM Sans', sans-serif;
    background: var(--bg);
    color: var(--text);
    min-height: 100vh;
  }

  /* HEADER */
  header {
    background: var(--navy);
    color: white;
    padding: 0;
    box-shadow: 0 2px 16px rgba(0,0,0,0.18);
  }
  .header-inner {
    max-width: 1100px;
    margin: 0 auto;
    padding: 20px 32px;
    display: flex;
    align-items: center;
    gap: 18px;
  }
  .header-icon {
    background: var(--gold);
    border-radius: 8px;
    padding: 8px 12px;
    font-size: 22px;
  }
  header h1 {
    font-family: 'Source Serif 4', serif;
    font-size: 1.55rem;
    font-weight: 600;
    letter-spacing: -0.02em;
  }
  header p {
    font-size: 0.82rem;
    opacity: 0.65;
    margin-top: 2px;
  }
  .badge {
    margin-left: auto;
    background: rgba(200,169,81,0.18);
    color: var(--gold-light);
    border: 1px solid rgba(200,169,81,0.35);
    border-radius: 20px;
    padding: 4px 14px;
    font-size: 0.77rem;
    font-weight: 500;
    letter-spacing: 0.04em;
    white-space: nowrap;
  }

  /* PROGRESS STEPS */
  .steps {
    background: white;
    border-bottom: 1px solid var(--border);
  }
  .steps-inner {
    max-width: 1100px;
    margin: 0 auto;
    padding: 0 32px;
    display: flex;
  }
  .step-btn {
    padding: 14px 20px;
    font-family: 'DM Sans', sans-serif;
    font-size: 0.86rem;
    font-weight: 500;
    color: var(--text-light);
    background: none;
    border: none;
    border-bottom: 3px solid transparent;
    cursor: pointer;
    transition: all 0.2s;
    display: flex;
    align-items: center;
    gap: 8px;
  }
  .step-btn .num {
    width: 22px; height: 22px;
    border-radius: 50%;
    background: var(--border);
    color: var(--text-light);
    font-size: 0.75rem;
    display: flex; align-items: center; justify-content: center;
    font-weight: 600;
    transition: all 0.2s;
  }
  .step-btn.active {
    color: var(--navy);
    border-bottom-color: var(--gold);
  }
  .step-btn.active .num {
    background: var(--navy);
    color: white;
  }
  .step-btn.done .num {
    background: var(--success);
    color: white;
  }

  /* MAIN LAYOUT */
  .main {
    max-width: 1100px;
    margin: 0 auto;
    padding: 32px;
  }

  /* PANELS */
  .panel { display: none; }
  .panel.active { display: block; }

  /* FORM STYLES */
  .card {
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: var(--radius);
    box-shadow: var(--shadow);
    padding: 28px 32px;
    margin-bottom: 24px;
  }
  .card-title {
    font-family: 'Source Serif 4', serif;
    font-size: 1.1rem;
    font-weight: 600;
    color: var(--navy);
    margin-bottom: 20px;
    padding-bottom: 12px;
    border-bottom: 1px solid var(--border);
  }
  .form-grid {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 16px;
  }
  .form-group {
    display: flex;
    flex-direction: column;
    gap: 6px;
  }
  .form-group.full { grid-column: 1 / -1; }
  label {
    font-size: 0.8rem;
    font-weight: 600;
    color: var(--text-light);
    text-transform: uppercase;
    letter-spacing: 0.05em;
  }
  input[type="text"], input[type="date"] {
    font-family: 'DM Sans', sans-serif;
    font-size: 0.95rem;
    padding: 10px 14px;
    border: 1.5px solid var(--border);
    border-radius: 7px;
    color: var(--text);
    background: #fafbfc;
    transition: border-color 0.2s, box-shadow 0.2s;
    outline: none;
  }
  input[type="text"]:focus, input[type="date"]:focus {
    border-color: var(--navy-light);
    box-shadow: 0 0 0 3px rgba(26,58,92,0.08);
    background: white;
  }

  /* UPLOAD ZONE */
  .upload-zone {
    border: 2px dashed var(--border);
    border-radius: var(--radius);
    padding: 40px 24px;
    text-align: center;
    cursor: pointer;
    transition: all 0.2s;
    background: #fafbfc;
  }
  .upload-zone:hover, .upload-zone.dragover {
    border-color: var(--navy-light);
    background: rgba(26,58,92,0.04);
  }
  .upload-zone .icon { font-size: 2.5rem; margin-bottom: 12px; }
  .upload-zone p { color: var(--text-light); font-size: 0.9rem; }
  .upload-zone strong { color: var(--navy); }
  #file-input { display: none; }

  /* IMAGE GRID */
  #image-pool {
    display: grid;
    grid-template-columns: repeat(auto-fill, minmax(120px, 1fr));
    gap: 12px;
    margin-top: 20px;
  }
  .img-card {
    border: 2px solid var(--border);
    border-radius: 8px;
    overflow: hidden;
    cursor: grab;
    transition: transform 0.15s, box-shadow 0.15s, border-color 0.15s;
    position: relative;
    background: #f0f3f8;
  }
  .img-card:hover { transform: translateY(-2px); box-shadow: 0 6px 20px rgba(0,0,0,0.12); }
  .img-card.dragging { opacity: 0.4; }
  .img-card img {
    width: 100%;
    aspect-ratio: 1;
    object-fit: cover;
    display: block;
  }
  .img-card .img-label {
    font-size: 0.68rem;
    padding: 4px 6px;
    color: var(--text-light);
    text-align: center;
    white-space: nowrap;
    overflow: hidden;
    text-overflow: ellipsis;
  }
  .img-card .remove-img {
    position: absolute;
    top: 4px; right: 4px;
    background: rgba(0,0,0,0.55);
    color: white;
    border: none;
    border-radius: 50%;
    width: 20px; height: 20px;
    font-size: 12px;
    cursor: pointer;
    display: none;
    align-items: center; justify-content: center;
    line-height: 1;
  }
  .img-card:hover .remove-img { display: flex; }
  .img-card.in-tab { border-color: var(--gold); }
  .img-card .tab-badge {
    position: absolute;
    bottom: 26px; left: 4px;
    background: var(--navy);
    color: white;
    font-size: 0.6rem;
    padding: 1px 5px;
    border-radius: 3px;
    font-weight: 600;
    display: none;
  }
  .img-card.in-tab .tab-badge { display: block; }

  /* FILE ICON (non-image uploads) */
  .file-icon-card {
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
    aspect-ratio: 1;
    width: 100%;
    background: #e8edf5;
    font-size: 2rem;
    gap: 4px;
  }
  .file-icon-card .file-ext {
    font-size: 0.55rem;
    font-weight: 700;
    text-transform: uppercase;
    color: var(--navy);
    background: rgba(26,58,92,0.15);
    padding: 1px 5px;
    border-radius: 2px;
  }

  /* TABS PANEL */
  .tabs-layout {
    display: grid;
    grid-template-columns: 280px 1fr;
    gap: 20px;
    align-items: start;
  }
  .tabs-sidebar {
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: var(--radius);
    box-shadow: var(--shadow);
    overflow: hidden;
  }
  .tabs-sidebar-header {
    background: var(--navy);
    color: white;
    padding: 14px 18px;
    font-size: 0.85rem;
    font-weight: 600;
    letter-spacing: 0.03em;
    display: flex;
    align-items: center;
    justify-content: space-between;
  }
  #tab-list { padding: 8px; }
  .tab-item {
    display: flex;
    align-items: center;
    gap: 10px;
    padding: 10px 12px;
    border-radius: 7px;
    cursor: pointer;
    transition: background 0.15s;
    border: 1.5px solid transparent;
    margin-bottom: 4px;
  }
  .tab-item:hover { background: var(--bg); }
  .tab-item.selected {
    background: rgba(26,58,92,0.07);
    border-color: var(--navy);
  }
  .tab-item .tab-num {
    width: 28px; height: 28px;
    border-radius: 6px;
    background: var(--navy);
    color: white;
    font-size: 0.82rem;
    font-weight: 700;
    display: flex; align-items: center; justify-content: center;
    flex-shrink: 0;
  }
  .tab-item .tab-name {
    font-size: 0.88rem;
    flex: 1;
    font-weight: 500;
  }
  .tab-item .tab-count {
    font-size: 0.75rem;
    color: var(--text-light);
    background: var(--bg);
    padding: 2px 8px;
    border-radius: 10px;
  }
  .tab-item .delete-tab {
    background: none;
    border: none;
    color: var(--text-light);
    cursor: pointer;
    font-size: 14px;
    padding: 2px;
    border-radius: 4px;
    opacity: 0;
    transition: opacity 0.15s, color 0.15s;
  }
  .tab-item:hover .delete-tab { opacity: 1; }
  .tab-item .delete-tab:hover { color: var(--danger); }

  #add-tab-btn {
    width: calc(100% - 16px);
    margin: 4px 8px 8px;
    padding: 9px;
    background: rgba(26,58,92,0.06);
    border: 1.5px dashed var(--navy-light);
    border-radius: 7px;
    color: var(--navy);
    font-family: 'DM Sans', sans-serif;
    font-size: 0.85rem;
    font-weight: 500;
    cursor: pointer;
    transition: background 0.15s;
  }
  #add-tab-btn:hover { background: rgba(26,58,92,0.12); }

  /* TAB DETAIL */
  .tab-detail {
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: var(--radius);
    box-shadow: var(--shadow);
    min-height: 400px;
  }
  .tab-detail-header {
    padding: 18px 24px;
    border-bottom: 1px solid var(--border);
    display: flex;
    align-items: center;
    gap: 16px;
  }
  .tab-detail-header .tab-num-big {
    width: 40px; height: 40px;
    border-radius: 10px;
    background: var(--navy);
    color: white;
    font-size: 1.1rem;
    font-weight: 700;
    display: flex; align-items: center; justify-content: center;
    flex-shrink: 0;
  }
  .tab-detail-header input[type="text"] {
    flex: 1;
    font-size: 1rem;
    font-weight: 600;
    padding: 8px 12px;
  }
  .tab-detail-body { padding: 20px 24px; }

  /* Drop zone inside tab */
  .tab-drop-zone {
    min-height: 160px;
    border: 2px dashed var(--border);
    border-radius: 8px;
    display: flex;
    align-items: center;
    justify-content: center;
    flex-direction: column;
    gap: 8px;
    color: var(--text-light);
    font-size: 0.85rem;
    transition: border-color 0.2s, background 0.2s;
    margin-bottom: 16px;
    padding: 16px;
  }
  .tab-drop-zone.dragover {
    border-color: var(--gold);
    background: rgba(200,169,81,0.06);
  }
  .tab-drop-zone.has-images { min-height: auto; padding: 0; border: none; }

  #tab-images-grid {
    display: grid;
    grid-template-columns: repeat(auto-fill, minmax(100px, 1fr));
    gap: 10px;
  }

  /* BUTTONS */
  .btn {
    font-family: 'DM Sans', sans-serif;
    font-size: 0.9rem;
    font-weight: 600;
    padding: 11px 24px;
    border: none;
    border-radius: 8px;
    cursor: pointer;
    transition: all 0.18s;
    display: inline-flex;
    align-items: center;
    gap: 8px;
  }
  .btn-primary {
    background: var(--navy);
    color: white;
  }
  .btn-primary:hover { background: var(--navy-light); transform: translateY(-1px); box-shadow: 0 4px 14px rgba(26,58,92,0.25); }
  .btn-gold {
    background: var(--gold);
    color: var(--navy);
  }
  .btn-gold:hover { background: var(--gold-light); transform: translateY(-1px); box-shadow: 0 4px 14px rgba(200,169,81,0.35); }
  .btn-outline {
    background: white;
    color: var(--navy);
    border: 1.5px solid var(--border);
  }
  .btn-outline:hover { border-color: var(--navy); background: var(--bg); }
  .btn:disabled { opacity: 0.5; cursor: not-allowed; transform: none !important; }

  .btn-row {
    display: flex;
    gap: 12px;
    align-items: center;
    margin-top: 28px;
  }

  /* REVIEW PANEL */
  .review-section { margin-bottom: 24px; }
  .review-label {
    font-size: 0.78rem;
    font-weight: 700;
    color: var(--text-light);
    text-transform: uppercase;
    letter-spacing: 0.06em;
    margin-bottom: 8px;
  }
  .review-value {
    font-size: 0.95rem;
    color: var(--text);
  }
  .review-tabs { display: flex; flex-direction: column; gap: 8px; }
  .review-tab-row {
    display: flex;
    align-items: center;
    gap: 12px;
    padding: 10px 16px;
    background: var(--bg);
    border-radius: 7px;
    font-size: 0.88rem;
  }
  .review-tab-row .rnum {
    font-weight: 700;
    color: var(--navy);
    min-width: 20px;
  }
  .review-tab-row .rtitle { flex: 1; }
  .review-tab-row .rcount {
    color: var(--text-light);
    font-size: 0.8rem;
  }

  /* GENERATE PANEL */
  .generate-center {
    text-align: center;
    padding: 40px 20px;
  }
  .generate-center .icon { font-size: 3.5rem; margin-bottom: 16px; }
  .generate-center h2 {
    font-family: 'Source Serif 4', serif;
    font-size: 1.4rem;
    color: var(--navy);
    margin-bottom: 8px;
  }
  .generate-center p { color: var(--text-light); font-size: 0.9rem; margin-bottom: 28px; }
  .spinner {
    width: 48px; height: 48px;
    border: 4px solid var(--border);
    border-top-color: var(--navy);
    border-radius: 50%;
    animation: spin 0.8s linear infinite;
    margin: 20px auto;
    display: none;
  }
  @keyframes spin { to { transform: rotate(360deg); } }

  /* UPLOAD PROGRESS */
  .upload-progress {
    margin-top: 12px;
    display: none;
  }
  .progress-bar {
    height: 4px;
    background: var(--border);
    border-radius: 2px;
    overflow: hidden;
  }
  .progress-fill {
    height: 100%;
    background: var(--navy);
    width: 0%;
    transition: width 0.3s;
  }
  .upload-status { font-size: 0.8rem; color: var(--text-light); margin-top: 6px; }

  /* NOTIFICATION */
  .notif {
    position: fixed;
    bottom: 24px; right: 24px;
    background: var(--navy);
    color: white;
    padding: 12px 20px;
    border-radius: 8px;
    font-size: 0.88rem;
    box-shadow: 0 6px 24px rgba(0,0,0,0.2);
    z-index: 999;
    transform: translateY(80px);
    opacity: 0;
    transition: all 0.3s;
  }
  .notif.show { transform: translateY(0); opacity: 1; }
  .notif.error { background: var(--danger); }

  /* SECTION INFO BOX */
  .info-box {
    background: rgba(26,58,92,0.06);
    border-left: 3px solid var(--navy);
    border-radius: 0 7px 7px 0;
    padding: 12px 16px;
    font-size: 0.84rem;
    color: var(--navy);
    margin-bottom: 20px;
  }

  /* RESPONSIVE */
  @media (max-width: 720px) {
    .main { padding: 16px; }
    .form-grid { grid-template-columns: 1fr; }
    .tabs-layout { grid-template-columns: 1fr; }
  }
</style>
</head>
<body>

<header>
  <div class="header-inner">
    <div class="header-icon">⚖️</div>
    <div>
      <h1>LTB Evidence Brief Generator</h1>
      <p>Landlord and Tenant Board — Hearing Preparation Tool</p>
    </div>
  </div>
</header>

<!-- Progress steps -->
<nav class="steps">
  <div class="steps-inner">
    <button class="step-btn active" onclick="goToStep(0)" id="step-btn-0">
      <span class="num">1</span> Case Information
    </button>
    <button class="step-btn" onclick="goToStep(1)" id="step-btn-1">
      <span class="num">2</span> Upload Files
    </button>
    <button class="step-btn" onclick="goToStep(2)" id="step-btn-2">
      <span class="num">3</span> Arrange Tabs
    </button>
    <button class="step-btn" onclick="goToStep(3)" id="step-btn-3">
      <span class="num">4</span> Review &amp; Generate
    </button>
  </div>
</nav>

<main class="main">

  <!-- STEP 1: CASE INFORMATION -->
  <div class="panel active" id="panel-0">
    <div class="card">
      <div class="card-title">Case Information</div>
      <div class="info-box">
        Enter the party and file details exactly as they appear on your LTB documents.
      </div>
      <div class="form-grid">
        <div class="form-group">
          <label>LTB File Number</label>
          <input type="text" id="file-number" placeholder="e.g. LTB-12345-24-OE">
        </div>
        <div class="form-group">
          <label>Hearing Date</label>
          <input type="date" id="hearing-date">
        </div>
        <div class="form-group">
          <label>Applicant Name</label>
          <input type="text" id="applicant-name" placeholder="Full legal name">
        </div>
        <div class="form-group">
          <label>Respondent Name</label>
          <input type="text" id="respondent-name" placeholder="Full legal name">
        </div>
        <div class="form-group full">
          <label>Applicant Address</label>
          <input type="text" id="applicant-address" placeholder="Street, City, Province, Postal Code">
        </div>
        <div class="form-group full">
          <label>Respondent / Rental Unit Address</label>
          <input type="text" id="respondent-address" placeholder="Street, City, Province, Postal Code">
        </div>
      </div>
    </div>
    <div class="btn-row">
      <button class="btn btn-primary" onclick="goToStep(1)">Next: Upload Photos →</button>
    </div>
  </div>

  <!-- STEP 2: UPLOAD FILES -->
  <div class="panel" id="panel-1">
    <div class="card">
      <div class="card-title">Upload Files</div>
      <div class="info-box">
        Upload all files you want to include. Images will be embedded in the PDF; other documents will be listed by name. You'll organize them into tabs in the next step.
      </div>

      <div class="upload-zone" id="upload-zone" onclick="document.getElementById('file-input').click()">
        <div class="icon">📁</div>
        <p><strong>Click to upload</strong> or drag &amp; drop files here</p>
        <p style="font-size:0.78rem; margin-top:4px;">All file types supported · Multiple files · Max 50 MB total</p>
      </div>
      <input type="file" id="file-input" multiple>

      <div class="upload-progress" id="upload-progress">
        <div class="progress-bar"><div class="progress-fill" id="progress-fill"></div></div>
        <div class="upload-status" id="upload-status">Uploading…</div>
      </div>

      <div id="image-pool"></div>
    </div>
    <div class="btn-row">
      <button class="btn btn-outline" onclick="goToStep(0)">← Back</button>
      <button class="btn btn-primary" onclick="proceedToTabs()">Next: Arrange Tabs →</button>

    </div>
  </div>

  <!-- STEP 3: ARRANGE TABS -->
  <div class="panel" id="panel-2">
    <div class="card">
      <div class="card-title">Arrange Tabs</div>
      <div class="info-box">
        Add tabs and drag photos from the pool below into each tab. Tabs appear in the order listed.
        You can rename each tab and reorder photos within a tab.
      </div>
      <div class="tabs-layout">
        <!-- Sidebar: tab list -->
        <div class="tabs-sidebar">
          <div class="tabs-sidebar-header">
            <span>TABS</span>
            <span id="tab-count-badge" style="background:rgba(255,255,255,0.2);padding:2px 8px;border-radius:10px;font-size:0.78rem;">0</span>
          </div>
          <div id="tab-list"></div>
          <button id="add-tab-btn" onclick="addTab()">+ Add Tab</button>
        </div>

        <!-- Detail: selected tab -->
        <div class="tab-detail" id="tab-detail">
          <div style="padding:40px;text-align:center;color:var(--text-light);font-size:0.9rem;">
            ← Select a tab to add photos, or create your first tab
          </div>
        </div>
      </div>

      <!-- Pool of all uploaded images -->
      <div style="margin-top:24px;">
        <div class="card-title" style="margin-bottom:12px;">File Pool — drag files into a tab above</div>
        <div id="pool-grid" style="display:grid;grid-template-columns:repeat(auto-fill,minmax(100px,1fr));gap:10px;"></div>
        <p id="pool-empty" style="color:var(--text-light);font-size:0.85rem;display:none;">All files have been added to tabs.</p>
      </div>
    </div>
    <div class="btn-row">
      <button class="btn btn-outline" onclick="goToStep(1)">← Back</button>
      <button class="btn btn-primary" onclick="goToStep(3)">Review &amp; Generate →</button>
    </div>
  </div>

  <!-- STEP 4: REVIEW & GENERATE -->
  <div class="panel" id="panel-3">
    <div class="card">
      <div class="card-title">Review Brief</div>

      <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;margin-bottom:24px;">
        <div>
          <div class="review-label">File Number</div>
          <div class="review-value" id="rev-file">—</div>
        </div>
        <div>
          <div class="review-label">Hearing Date</div>
          <div class="review-value" id="rev-date">—</div>
        </div>
        <div>
          <div class="review-label">Applicant</div>
          <div class="review-value" id="rev-applicant">—</div>
        </div>
        <div>
          <div class="review-label">Respondent</div>
          <div class="review-value" id="rev-respondent">—</div>
        </div>
        <div>
          <div class="review-label">Applicant Address</div>
          <div class="review-value" id="rev-app-addr">—</div>
        </div>
        <div>
          <div class="review-label">Respondent Address</div>
          <div class="review-value" id="rev-res-addr">—</div>
        </div>
      </div>

      <div class="review-label">Tabs</div>
      <div class="review-tabs" id="rev-tabs"></div>
    </div>

    <div class="card">
      <div class="generate-center">
        <div class="icon">📄</div>
        <h2>Ready to generate your Evidence Brief</h2>
        <p>Your PDF will include a title page, table of contents, and all photo tabs with page numbers.</p>
        <button class="btn btn-gold" id="generate-btn" onclick="generatePDF()" style="font-size:1rem;padding:14px 36px;">
          ⚡ Generate PDF
        </button>
        <div class="spinner" id="spinner"></div>
        <p id="gen-status" style="margin-top:12px;display:none;"></p>
      </div>
    </div>

    <div class="btn-row">
      <button class="btn btn-outline" onclick="goToStep(2)">← Back to Tabs</button>
    </div>
  </div>

</main>

<!-- Notification -->
<div class="notif" id="notif"></div>

<script>
// ─── STATE ────────────────────────────────────────────────────────────────────
let uploadedImages = [];  // [{id, thumb, original}]
let tabs = [];            // [{id, title, imageIds:[]}]
let selectedTabId = null;
let currentStep = 0;

// ─── NAVIGATION ──────────────────────────────────────────────────────────────
function goToStep(step) {
  document.querySelectorAll('.panel').forEach((p, i) => p.classList.toggle('active', i === step));
  document.querySelectorAll('.step-btn').forEach((b, i) => {
    b.classList.toggle('active', i === step);
    if (i < step) b.querySelector('.num').textContent = '✓', b.classList.add('done');
    else b.querySelector('.num').textContent = i + 1, b.classList.remove('done');
  });
  currentStep = step;
  if (step === 2) renderTabPanel();
  if (step === 3) renderReview();
}

// ─── NOTIFICATIONS ───────────────────────────────────────────────────────────
function notify(msg, isError = false) {
  const el = document.getElementById('notif');
  el.textContent = msg;
  el.className = 'notif show' + (isError ? ' error' : '');
  setTimeout(() => el.classList.remove('show'), 3000);
}

// ─── UPLOAD ──────────────────────────────────────────────────────────────────
const uploadZone = document.getElementById('upload-zone');
uploadZone.addEventListener('dragover', e => { e.preventDefault(); uploadZone.classList.add('dragover'); });
uploadZone.addEventListener('dragleave', () => uploadZone.classList.remove('dragover'));
uploadZone.addEventListener('drop', e => {
  e.preventDefault();
  uploadZone.classList.remove('dragover');
  handleFiles(e.dataTransfer.files);
});
document.getElementById('file-input').addEventListener('change', e => handleFiles(e.target.files));

async function handleFiles(files) {
  if (!files.length) return;
  const formData = new FormData();
  for (const f of files) formData.append('files', f);

  document.getElementById('upload-progress').style.display = 'block';
  document.getElementById('progress-fill').style.width = '30%';
  document.getElementById('upload-status').textContent = `Uploading ${files.length} file(s)…`;

  try {
    const res = await fetch('/upload', { method: 'POST', body: formData });
    document.getElementById('progress-fill').style.width = '100%';
    const data = await res.json();
    if (data.error) throw new Error(data.error);

    uploadedImages.push(...data.files);
    renderImagePool();
    notify(`✓ ${data.files.length} file(s) uploaded`);
  } catch (e) {
    notify('Upload failed: ' + e.message, true);
  } finally {
    setTimeout(() => {
      document.getElementById('upload-progress').style.display = 'none';
      document.getElementById('progress-fill').style.width = '0%';
    }, 800);
    document.getElementById('file-input').value = '';
  }
}

function renderImagePool() {
  const pool = document.getElementById('image-pool');
  pool.innerHTML = '';
  uploadedImages.forEach((img, idx) => {
    const card = document.createElement('div');
    card.className = 'img-card';
    card.dataset.id = img.id;
    const assignedTab = tabs.find(t => t.imageIds.includes(img.id));
    if (assignedTab) {
      card.classList.add('in-tab');
    }
    const mediaHtml0 = img.is_image
      ? `<img src="/thumbnail/${img.thumb || img.original}" alt="File ${idx+1}" loading="lazy">`
      : `<div class="file-icon-card"><span>📄</span><span class="file-ext">${(img.name||img.id).split('.').pop().toLowerCase()}</span></div>`;
    card.innerHTML = `
      ${mediaHtml0}
      <div class="tab-badge">${assignedTab ? 'Tab ' + (tabs.indexOf(assignedTab)+1) : ''}</div>
      <button class="remove-img" onclick="removeImage('${img.id}',event)" title="Remove">×</button>
      <div class="img-label">${img.name || ('File ' + (idx+1))}</div>
    `;
    pool.appendChild(card);
  });
}

function removeImage(id, e) {
  e.stopPropagation();
  uploadedImages = uploadedImages.filter(i => i.id !== id);
  tabs.forEach(t => { t.imageIds = t.imageIds.filter(i => i !== id); });
  renderImagePool();
  if (currentStep === 2) renderTabPanel();
}

// ─── PROCEED TO TABS ─────────────────────────────────────────────────────────
function proceedToTabs() {
  if (uploadedImages.length === 0) {
    notify('Please upload at least one file first.', true);
    return;
  }
  if (tabs.length === 0) addTab();
  goToStep(2);
}

// ─── TABS PANEL ──────────────────────────────────────────────────────────────
function addTab() {
  const tab = { id: 'tab_' + Date.now(), title: '', imageIds: [] };
  tabs.push(tab);
  selectedTabId = tab.id;
  renderTabPanel();
}

function deleteTab(id, e) {
  e.stopPropagation();
  tabs = tabs.filter(t => t.id !== id);
  if (selectedTabId === id) selectedTabId = tabs.length ? tabs[tabs.length - 1].id : null;
  renderTabPanel();
}

function selectTab(id) {
  selectedTabId = id;
  renderTabPanel();
}

function renderTabPanel() {
  // Tab count badge
  document.getElementById('tab-count-badge').textContent = tabs.length;

  // Tab list sidebar
  const list = document.getElementById('tab-list');
  list.innerHTML = '';
  tabs.forEach((tab, idx) => {
    const item = document.createElement('div');
    item.className = 'tab-item' + (tab.id === selectedTabId ? ' selected' : '');
    item.onclick = () => selectTab(tab.id);
    item.innerHTML = `
      <div class="tab-num">${idx + 1}</div>
      <div class="tab-name">${tab.title || `Tab ${idx+1}`}</div>
      <div class="tab-count">${tab.imageIds.length} 📎</div>
      <button class="delete-tab" onclick="deleteTab('${tab.id}',event)" title="Delete tab">🗑</button>
    `;
    list.appendChild(item);
  });

  // Tab detail
  const detail = document.getElementById('tab-detail');
  if (!selectedTabId || !tabs.find(t => t.id === selectedTabId)) {
    detail.innerHTML = `<div style="padding:40px;text-align:center;color:var(--text-light);font-size:0.9rem;">← Select a tab or click "Add Tab"</div>`;
  } else {
    const tab = tabs.find(t => t.id === selectedTabId);
    const tabIdx = tabs.indexOf(tab) + 1;
    detail.innerHTML = `
      <div class="tab-detail-header">
        <div class="tab-num-big">${tabIdx}</div>
        <input type="text" placeholder="Tab title (optional, e.g. 'Mould in Bathroom')"
          value="${tab.title}"
          oninput="tab_title_update(this.value)"
          style="flex:1;">
      </div>
      <div class="tab-detail-body">
        <p style="font-size:0.82rem;color:var(--text-light);margin-bottom:12px;">
          Drag files from the pool below into this area, or click files to add/remove them.
        </p>
        <div class="tab-drop-zone ${tab.imageIds.length ? 'has-images' : ''}" id="tab-drop-zone"
          ondragover="dzOver(event)" ondragleave="dzLeave(event)" ondrop="dzDrop(event,'${tab.id}')">
          ${tab.imageIds.length === 0
            ? '<div style="font-size:1.8rem">📥</div><div>Drop files here</div>'
            : ''}
          <div id="tab-images-grid">
            ${tab.imageIds.map((imgId, i) => {
              const img = uploadedImages.find(u => u.id === imgId);
              if (!img) return '';
              const mediaHtml = img.is_image
                ? `<img src="/thumbnail/${img.thumb || img.original}" loading="lazy">`
                : `<div class="file-icon-card"><span>📄</span><span class="file-ext">${(img.name||img.id).split('.').pop().toLowerCase()}</span></div>`;
              return `
                <div class="img-card" draggable="true"
                  ondragstart="imgDragStart(event,'${imgId}')"
                  title="Click to remove from this tab">
                  ${mediaHtml}
                  <button class="remove-img" onclick="removeFromTab('${tab.id}','${imgId}',event)" title="Remove">×</button>
                  <div class="img-label">${img.name || ('File ' + (i+1))}</div>
                </div>`;
            }).join('')}
          </div>
        </div>
      </div>
    `;
  }

  // Pool grid
  renderPoolGrid();
}

function tab_title_update(val) {
  const tab = tabs.find(t => t.id === selectedTabId);
  if (tab) tab.title = val;
  // Update sidebar label
  const list = document.getElementById('tab-list');
  const items = list.querySelectorAll('.tab-item');
  const idx = tabs.indexOf(tab);
  if (items[idx]) {
    items[idx].querySelector('.tab-name').textContent = val || `Tab ${idx+1}`;
  }
}

function renderPoolGrid() {
  const grid = document.getElementById('pool-grid');
  const unassigned = uploadedImages.filter(img =>
    !tabs.some(t => t.imageIds.includes(img.id))
  );

  document.getElementById('pool-empty').style.display = unassigned.length === 0 && uploadedImages.length > 0 ? 'block' : 'none';

  grid.innerHTML = unassigned.map((img, idx) => {
    const mediaHtml = img.is_image
      ? `<img src="/thumbnail/${img.thumb || img.original}" loading="lazy">`
      : `<div class="file-icon-card"><span>📄</span><span class="file-ext">${(img.name||img.id).split('.').pop().toLowerCase()}</span></div>`;
    return `
      <div class="img-card" draggable="true"
        ondragstart="imgDragStart(event,'${img.id}')"
        onclick="addToSelectedTab('${img.id}')"
        title="Click to add to selected tab / Drag to a tab">
        ${mediaHtml}
        <div class="img-label">${img.name || ('File ' + (uploadedImages.indexOf(img)+1))}</div>
      </div>`;
  }).join('');
}

// ─── DRAG AND DROP ───────────────────────────────────────────────────────────
let draggingId = null;

function imgDragStart(e, imgId) {
  draggingId = imgId;
  e.dataTransfer.effectAllowed = 'move';
  e.dataTransfer.setData('text/plain', imgId);
}

function dzOver(e) {
  e.preventDefault();
  document.getElementById('tab-drop-zone').classList.add('dragover');
}
function dzLeave(e) {
  document.getElementById('tab-drop-zone').classList.remove('dragover');
}
function dzDrop(e, tabId) {
  e.preventDefault();
  document.getElementById('tab-drop-zone').classList.remove('dragover');
  const imgId = e.dataTransfer.getData('text/plain') || draggingId;
  if (!imgId) return;
  addImageToTab(tabId, imgId);
}

function addImageToTab(tabId, imgId) {
  const tab = tabs.find(t => t.id === tabId);
  if (!tab) return;
  // Remove from any other tab first
  tabs.forEach(t => { t.imageIds = t.imageIds.filter(i => i !== imgId); });
  if (!tab.imageIds.includes(imgId)) tab.imageIds.push(imgId);
  renderTabPanel();
}

function addToSelectedTab(imgId) {
  if (!selectedTabId) { notify('Select a tab first.', true); return; }
  addImageToTab(selectedTabId, imgId);
}

function removeFromTab(tabId, imgId, e) {
  e.stopPropagation();
  const tab = tabs.find(t => t.id === tabId);
  if (tab) tab.imageIds = tab.imageIds.filter(i => i !== imgId);
  renderTabPanel();
}

// ─── REVIEW ──────────────────────────────────────────────────────────────────
function renderReview() {
  document.getElementById('rev-file').textContent = document.getElementById('file-number').value || '—';
  const d = document.getElementById('hearing-date').value;
  document.getElementById('rev-date').textContent = d ? new Date(d + 'T00:00').toLocaleDateString('en-CA', {year:'numeric',month:'long',day:'numeric'}) : '—';
  document.getElementById('rev-applicant').textContent = document.getElementById('applicant-name').value || '—';
  document.getElementById('rev-respondent').textContent = document.getElementById('respondent-name').value || '—';
  document.getElementById('rev-app-addr').textContent = document.getElementById('applicant-address').value || '—';
  document.getElementById('rev-res-addr').textContent = document.getElementById('respondent-address').value || '—';

  const revTabs = document.getElementById('rev-tabs');
  if (tabs.length === 0) {
    revTabs.innerHTML = '<p style="color:var(--text-light);font-size:0.85rem;">No tabs created yet.</p>';
  } else {
    revTabs.innerHTML = tabs.map((t, i) => `
      <div class="review-tab-row">
        <div class="rnum">${i+1}</div>
        <div class="rtitle">${t.title || `Tab ${i+1}`}</div>
        <div class="rcount">${t.imageIds.length} file${t.imageIds.length !== 1 ? 's' : ''}</div>
      </div>
    `).join('');
  }
}

// ─── GENERATE PDF ────────────────────────────────────────────────────────────
async function generatePDF() {
  if (tabs.length === 0) { notify('Add at least one tab first.', true); return; }

  const btn = document.getElementById('generate-btn');
  const spinner = document.getElementById('spinner');
  const status = document.getElementById('gen-status');

  btn.disabled = true;
  spinner.style.display = 'block';
  status.style.display = 'block';
  status.textContent = 'Building your Evidence Brief…';
  status.style.color = 'var(--text-light)';

  const payload = {
    case_info: {
      file_number: document.getElementById('file-number').value,
      hearing_date: document.getElementById('hearing-date').value,
      applicant_name: document.getElementById('applicant-name').value,
      respondent_name: document.getElementById('respondent-name').value,
      applicant_address: document.getElementById('applicant-address').value,
      respondent_address: document.getElementById('respondent-address').value,
    },
    tabs: tabs.map(t => ({
      title: t.title,
      images: t.imageIds,
    }))
  };

  try {
    const res = await fetch('/generate', {
      method: 'POST',
      headers: { 'Content-Type': 'application/json' },
      body: JSON.stringify(payload)
    });

    if (!res.ok) {
      const err = await res.json();
      throw new Error(err.error || 'Server error');
    }

    const blob = await res.blob();
    const url = URL.createObjectURL(blob);
    const a = document.createElement('a');
    a.href = url;
    a.download = `Evidence_Brief_${payload.case_info.file_number || 'LTB'}.pdf`;
    a.click();
    URL.revokeObjectURL(url);

    status.textContent = '✓ PDF downloaded successfully!';
    status.style.color = 'var(--success)';
    notify('✓ Evidence Brief generated!');
  } catch (e) {
    status.textContent = 'Error: ' + e.message;
    status.style.color = 'var(--danger)';
    notify('Generation failed: ' + e.message, true);
  } finally {
    btn.disabled = false;
    spinner.style.display = 'none';
  }
}
</script>
</body>
</html>


In [ ]:
import os, sys, threading, time

# ── Detect environment ───────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Load the Flask app ───────────────────────────────────────────────
import importlib.util

spec = importlib.util.spec_from_file_location("ltb_app", "app.py")
mod = importlib.util.module_from_spec(spec)
mod.__file__ = os.path.abspath("app.py")   # lets os.path.dirname(__file__) resolve to CWD
spec.loader.exec_module(mod)
flask_app = mod.app

os.makedirs(flask_app.config['UPLOAD_FOLDER'], exist_ok=True)

# ── Start server in background thread ───────────────────────────────
def _run():
    flask_app.run(port=5050, debug=False, use_reloader=False, threaded=True)

t = threading.Thread(target=_run, daemon=True)
t.start()
time.sleep(1.5)

# ── Show access URL ──────────────────────────────────────────────────
if IN_COLAB:
    from google.colab.output import eval_js
    url = eval_js("google.colab.kernel.proxyPort(5050)")
    print(f"✅  App is running!  Open it here:\n    {url}")
else:
    print("✅  App is running!  Open it here:\n    http://localhost:5050")
